# Train on a Colab GPU

Runs `scripts/train.py` on a Colab GPU runtime instead of the local CPU. It clones the repo,
downloads the PhiUSIIL dataset from UCI, builds the splits, trains and evaluates the model,
then packs the weights and splits into one zip to download.

Connect this notebook to a Colab **GPU** runtime first, either in the VS Code Colab extension or
in the Colab web UI under *Runtime → Change runtime type*.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU type"
print(torch.cuda.get_device_name(0))

In [ ]:
REPO = "https://github.com/Kluey/URL-Phishing-Classifier.git"
BRANCH = "hostname-input"

%cd /content
!rm -rf URL-Phishing-Classifier && git clone -q --depth 1 -b {BRANCH} {REPO}
%cd /content/URL-Phishing-Classifier
!git log --oneline -1
!pip install -q -e ".[pretrain]"

In [ ]:
!mkdir -p data model
!wget -q -O /tmp/phiusiil.zip "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
!unzip -o -q /tmp/phiusiil.zip -d data && ls -la data

Builds the hostname splits from all balanced data and trains for 2 epochs (the defaults).

In [ ]:
!python scripts/train.py --prepare-data 2>&1 | tee model/train.log

In [ ]:
!python scripts/evaluate.py --split test
!python scripts/evaluate.py --split real_world

Packs everything needed to evaluate locally on the same splits: weights, run and metric files, splits.

In [ ]:
!cd /content/URL-Phishing-Classifier && zip -q -0 /content/phishing_classifier_run.zip model/phishing_classifier.pth model/*.json model/train.log data/train.csv data/validation.csv data/test.csv data/splits.json
!ls -la /content/phishing_classifier_run.zip

try:
    from google.colab import files
    files.download("/content/phishing_classifier_run.zip")
except Exception as exc:  # e.g. not supported from the current client
    print(f"Automatic download unavailable ({exc}); download /content/phishing_classifier_run.zip from the file browser.")